# Fight Matrix verification

**Source:** Fight Matrix (https://www.fightmatrix.com), historical and current MMA rankings

**Packages:** requests (fetch pages), pandas (pd.read_html for table parsing), BeautifulSoup (dropdown harvesting and fallback parsing)

Verify the rankings can be scraped for use as the primary ranking validation source.

**Sample:** women's strawweight, for continuity with the thin slice and the Trends / Wikipedia verifications.


## Section 1: Setup


In [ ]:
# BLOCK 1: Imports and configuration

!pip install beautifulsoup4 -q

import requests
import pandas as pd
import time
from datetime import datetime
from bs4 import BeautifulSoup
from io import StringIO

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Descriptive User Agent
USER_AGENT = "UFC-Value-Mapper/0.1 (MSc academic project; contact via GitHub th1555)"
HEADERS = {"User-Agent": USER_AGENT}

# Time between requests (seconds)
REQUEST_DELAY = 2

BASE = "https://www.fightmatrix.com"

print(f"User-Agent set: {USER_AGENT}")

User-Agent set: UFC-Value-Mapper/0.1 (MSc academic project; tomhorwood1@gmail.com)


In [ ]:
# BLOCK 2: Helper to fetch a page with error handling

def fetch(url, headers=HEADERS, timeout=15):
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        print(f"  GET {url}")
        print(f"    status: {r.status_code}, length: {len(r.text):,} chars")
        # Quick check: is it HTML, not a block page or empty?
        if r.status_code == 200 and '<table' not in r.text.lower() and 'select' not in r.text.lower():
            print("    WARN: 200 but no <table> or <select> found; page may be a block or redirect")
        return r.text, r.status_code
    except Exception as e:
        print(f"  GET {url} FAILED: {type(e).__name__}: {e}")
        return None, None

## Section 2: Current rankings scrape (warm-up)

Quick check: pull a current division ranking page, filtered to UFC via the OrgFilter parameter, and confirm the table parses.

In [ ]:
# BLOCK 3: Pull a ranking page

# Current-rankings URL pattern:
#   /mma-ranks/<division-slug>/?RF=FM&OrgFilter=UFC&...

current_url = f"{BASE}/mma-ranks/womens-strawweight/?RF=FM&OrgFilter=UFC"

html, status = fetch(current_url)
time.sleep(REQUEST_DELAY)

if html and status == 200:
    # parse all <table> elements
    try:
        tables = pd.read_html(StringIO(html))
        print(f"\n  pd.read_html found {len(tables)} table(s)")
        for j, t in enumerate(tables):
            print(f"    table {j}: shape {t.shape}, columns {list(t.columns)[:6]}")
    except Exception as e:
        print(f"  pd.read_html failed: {type(e).__name__}: {e}")
        tables = []
else:
    print("  Could not fetch current rankings")
    tables = []

  GET https://www.fightmatrix.com/mma-ranks/womens-strawweight/?RF=FM&OrgFilter=UFC
    status: 200, length: 203,214 chars

  pd.read_html found 47 table(s)
    table 0: shape (5, 5), columns [0, 1, 2, 3, 4]
    table 1: shape (44, 4), columns [0, 1, 2, 3]
    table 2: shape (3, 3), columns [0, 1, 2]
    table 3: shape (3, 3), columns [0, 1, 2]
    table 4: shape (3, 3), columns [0, 1, 2]
    table 5: shape (3, 3), columns [0, 1, 2]
    table 6: shape (3, 3), columns [0, 1, 2]
    table 7: shape (3, 3), columns [0, 1, 2]
    table 8: shape (3, 3), columns [0, 1, 2]
    table 9: shape (3, 3), columns [0, 1, 2]
    table 10: shape (3, 3), columns [0, 1, 2]
    table 11: shape (3, 3), columns [0, 1, 2]
    table 12: shape (3, 3), columns [0, 1, 2]
    table 13: shape (3, 3), columns [0, 1, 2]
    table 14: shape (3, 3), columns [0, 1, 2]
    table 15: shape (3, 3), columns [0, 1, 2]
    table 16: shape (3, 3), columns [0, 1, 2]
    table 17: shape (3, 3), columns [0, 1, 2]
    table 18: s

In [ ]:
# BLOCK 4: Ranking table check

ranking_table = None
if tables:
    for t in tables:
        cols_lower = [str(c).lower() for c in t.columns]
        # Ideal if the rankings table mentions 'fighter' and has >5 rows
        if any('fighter' in c for c in cols_lower) and len(t) > 5:
            ranking_table = t
            break
    # Backup: if no column named 'fighter', use the largest table
    if ranking_table is None:
        ranking_table = max(tables, key=len)
        print("  No 'Fighter' column found; using largest table as fallback.")

if ranking_table is not None:
    print(f"Ranking table shape: {ranking_table.shape}")
    print(f"Columns: {list(ranking_table.columns)}")
    print("\nTop 10:")
    print(ranking_table.head(10).to_string())
else:
    print("No ranking table found. BeautifulSoup fallback needed.")

  No 'Fighter' column found; using largest table as fallback.
Ranking table shape: (44, 4)
Columns: [0, 1, 2, 3]

Top 10:
                0                                                                                               1                                                                                               2                                                                                               3
0  Rank [Overall]                                                                                   Fighter (Age)                                                                                          Record                                                                                          Points
1           1 [1]   Weili Zhang (36)  26-4-0  383  Last Fight: 11/15/2025 [UFC] vs [#1 W125] Valentina Shevchenko   Weili Zhang (36)  26-4-0  383  Last Fight: 11/15/2025 [UFC] vs [#1 W125] Valentina Shevchenko   Weili Zhang (36)  26-4-0  383  Last Fight: 11/15/2025 

## Section 3: Historical access

The historical endpoint uses disguised codes:

/historical-mma-rankings/ranking-snapshots/?Issue=1005&Division=3

Issue=1005 maps to a date (01/04/2026) and Division=3 maps to a division name (Middleweight). The page itself contains the full mapping in two dropdown `<select>` elements. Both dropdowns are harvested to uncover and map the Issue and Division codes.

In [ ]:
# BLOCK 5: Harvest the Issue and Division dropdowns
#
# Load a historical snapshot page and parse the two <select> elements to build
# lookup tables:
#   issue_lookup:    {issue_id: date_string}
#   division_lookup: {division_id: division_name}

seed_url = f"{BASE}/historical-mma-rankings/ranking-snapshots/?Issue=1005&Division=3"
print("Fetching a seed historical page to harvest dropdowns:")
hist_html, hist_status = fetch(seed_url)
time.sleep(REQUEST_DELAY)

issue_lookup = {}
division_lookup = {}

if hist_html and hist_status == 200:
    soup = BeautifulSoup(hist_html, 'html.parser')

    # Find all <select> elements and identify which is Issue, which is Division.
    selects = soup.find_all('select')
    print(f"\n  Found {len(selects)} <select> element(s) on the page")

    for sel in selects:
        name_attr = (sel.get('name') or '').lower()
        options = sel.find_all('option')
        # Build {value: text} for this select
        opt_map = {}
        for opt in options:
            val = opt.get('value', '').strip()
            txt = opt.get_text(strip=True)
            if val:
                opt_map[val] = txt

        # Classification; Issue options look like dates; Division options look
        # like names
        sample_texts = list(opt_map.values())[:3]
        print(f"    select name='{name_attr}', {len(opt_map)} options, sample: {sample_texts}")

        if 'issue' in name_attr:
            issue_lookup = opt_map
        elif 'division' in name_attr:
            division_lookup = opt_map

    print(f"\n  issue_lookup: {len(issue_lookup)} entries")
    print(f"  division_lookup: {len(division_lookup)} entries")
    if division_lookup:
        print(f"\n  Divisions available:")
        for vid, vname in division_lookup.items():
            print(f"    {vid}: {vname}")
else:
    print("  Could not fetch the seed historical page.")

Fetching a seed historical page to harvest dropdowns:
  GET https://www.fightmatrix.com/historical-mma-rankings/ranking-snapshots/?Issue=1005&Division=3
    status: 200, length: 121,865 chars

  Found 2 <select> element(s) on the page
    select name='issue', 222 options, sample: ['05/03/2026', '04/05/2026', '03/01/2026']
    select name='division', 18 options, sample: ['Pound-for-Pound', 'Division Dominance List', 'Heavyweight']

  issue_lookup: 222 entries
  division_lookup: 18 entries

  Divisions available:
    -1: Pound-for-Pound
    11: Division Dominance List
    1: Heavyweight
    2: LightHeavyweight
    3: Middleweight
    4: Welterweight
    5: Lightweight
    6: Featherweight
    7: Bantamweight
    8: Flyweight
    9: Strawweight
    -2: Women Pound-for-Pound
    17: Women - Division Dominance
    16: Women - Featherweight+
    15: Women - Bantamweight
    14: Women - Flyweight
    13: Women - Strawweight
    12: Women - Atomweight


In [ ]:
# BLOCK 6: Find the women's strawweight division

# Go from (division name, date) > ranking table.

def find_division_code(lookup, name_contains):
    for vid, vname in lookup.items():
        if name_contains.lower() in vname.lower():
            return vid, vname
    return None, None

if division_lookup and issue_lookup:
    # Find women's strawweight
    sw_id, sw_name = find_division_code(division_lookup, 'strawweight')
    # Women specific
    women_sw_id, women_sw_name = find_division_code(division_lookup, "women - strawweight")
    if women_sw_id:
        sw_id, sw_name = women_sw_id, women_sw_name

    # Most recent issue
    recent_issue_id = max(issue_lookup.keys(), key=lambda x: int(x) if x.isdigit() else -1)
    recent_issue_date = issue_lookup[recent_issue_id]

    print(f"Selected division: {sw_id} ({sw_name})")
    print(f"Selected issue: {recent_issue_id} ({recent_issue_date})")
    print()

    if sw_id:
        hist_sample_url = f"{BASE}/historical-mma-rankings/ranking-snapshots/?Issue={recent_issue_id}&Division={sw_id}"
        print("Fetching historical women's strawweight snapshot:")
        snap_html, snap_status = fetch(hist_sample_url)
        time.sleep(REQUEST_DELAY)

        if snap_html and snap_status == 200:
            try:
                snap_tables = pd.read_html(StringIO(snap_html))
                # Identify the ranking table (Fighter column, many rows)
                hist_ranking = None
                for t in snap_tables:
                    cols_lower = [str(c).lower() for c in t.columns]
                    if any('fighter' in c for c in cols_lower) and len(t) > 5:
                        hist_ranking = t
                        break
                if hist_ranking is None and snap_tables:
                    hist_ranking = max(snap_tables, key=len)

                if hist_ranking is not None:
                    print(f"\n  Historical ranking table shape: {hist_ranking.shape}")
                    print(f"  Columns: {list(hist_ranking.columns)}")
                    print("\n  First 10 rows:")
                    print(hist_ranking.head(10).to_string())
                else:
                    print("  No ranking table found in the historical snapshot.")
            except Exception as e:
                print(f"  pd.read_html failed: {type(e).__name__}: {e}")
    else:
        print("  Could not find a strawweight division code in the lookup.")
else:
    print("  Lookups not built; cannot proceed. Check Cell 5 output.")

Selected division: 13 (Women - Strawweight)
Selected issue: 1022 (05/03/2026)

Fetching historical women's strawweight snapshot:
  GET https://www.fightmatrix.com/historical-mma-rankings/ranking-snapshots/?Issue=1022&Division=13
    status: 200, length: 121,212 chars

  Historical ranking table shape: (26, 4)
  Columns: [0, 1, 2, 3]

  First 10 rows:
      0    1                  2       3
0  Rank  ↑ ↓            Fighter  Points
1     1  NaN        Weili Zhang     383
2     2  NaN     Mackenzie Dern     264
3     3  NaN    Virna Jandiroba     260
4     4  NaN     Tatiana Suarez     230
5     5    1  Gillian Robertson     164
6     6   -1        Xiaonan Yan     162
7     7  NaN      Tabatha Ricci     132
8     8  NaN       Denise Gomes     128
9     9  NaN       Amanda Lemos     123


## Section 4: Name format check

Fighter name strings are compared against the UFCStats and Wikipedia formats. Name inconsistency recurs across all sources, so it must be handled early in the pipeline.

In [ ]:
# BLOCK 7: Check name format

thin_slice_names = [
    'Rose Namajunas', 'Ashley Yoder', 'Gloria de Paula', 'Tatiana Suarez',
    'Shauna Bannon', 'Jessica Penne', 'Amanda Ribas', 'Alexia Thainara',
    'Denise Gomes', 'Yan Xiaonan',
]

# Use the historical ranking table
name_source = None
try:
    if 'hist_ranking' in dir() and hist_ranking is not None:
        name_source = hist_ranking
        print("Using historical ranking table for name inspection.")
    elif ranking_table is not None:
        name_source = ranking_table
        print("Using current ranking table for name inspection.")
except NameError:
    pass

if name_source is not None:
    # Find the fighter name column
    fighter_col = None
    for c in name_source.columns:
        if 'fighter' in str(c).lower():
            fighter_col = c
            break
    if fighter_col is None:
        # Fallback guess (the column with the longest average string content)
        str_lengths = {c: name_source[c].astype(str).str.len().mean() for c in name_source.columns}
        fighter_col = max(str_lengths, key=str_lengths.get)
        print(f"No 'Fighter' column; guessing '{fighter_col}' by string length.")

    fm_names = name_source[fighter_col].astype(str).tolist()
    print(f"\nFight Matrix names (column '{fighter_col}'), first 15:")
    for n in fm_names[:15]:
        print(f"  {n}")

    # Direct overlap comparison group
    fm_names_set = set(n.strip() for n in fm_names)
    overlap = [n for n in thin_slice_names if n in fm_names_set]
    print(f"\nExact matches with thin-slice fighters: {len(overlap)}")
    print(f"  Matched: {overlap}")
else:
    print("No table available for name inspection.")

Using historical ranking table for name inspection.
No 'Fighter' column; guessing '2' by string length.

Fight Matrix names (column '2'), first 15:
  Fighter
  Weili Zhang
  Mackenzie Dern
  Virna Jandiroba
  Tatiana Suarez
  Gillian Robertson
  Xiaonan Yan
  Tabatha Ricci
  Denise Gomes
  Amanda Lemos
  Iasmin Lucindo
  Lupita Godinez
  Tecia Pennington
  Alexia Thainara
  Jingnan Xiong

Exact matches with thin-slice fighters: 4
  Matched: ['Tatiana Suarez', 'Amanda Ribas', 'Alexia Thainara', 'Denise Gomes']


## Section 5: Verification summary

**Current rankings**
- Page reachable, UFC-filtered: yes
- Table parsed by pd.read_html: yes
- Columns: Rank, movement, Fighter, Points

**Historical snapshots**
- Dropdown harvest: 222 issue dates, 18 divisions mapped
- Historical URL built from (division, date): yes
- Historical ranking table parsed: yes (Women - Strawweight, 05/03/2026, 25 ranked fighters)

**Name format**
- FM flips non-Western names to given name first (Yan Xiaonan listed as "Xiaonan Yan")
- Exact matches with thin-slice fighters: 4 of 5 present in snapshot; the miss (Yan Xiaonan) is a name-order flip, not a coverage gap
- Third source to hit the name-matching problem (after Trends and Wikipedia)

### Quirks, resolved in the validation

- Header row leaks into the data (no `<th>` tags); resolved in the validation by coercing rank and points to numeric, so the header row becomes NaN and drops.
- Date format is MM/DD/YYYY, confirmed in the validation by bracketing the freeze date (issue 1031 = 07/05/2026 sits 6 days before the 11 July freeze, so 5 July, not 7 May).
- Each division snapshot returns the top 25; the per-division validation uses these, so pagination beyond 25 was not required.
- UFC divisions are selected by division ID on the historical endpoint; OrgFilter is not needed there.

### Division codes (for the later scrape)

Men: 1 Heavyweight, 2 LightHeavyweight, 3 Middleweight, 4 Welterweight, 5 Lightweight, 6 Featherweight, 7 Bantamweight, 8 Flyweight, 9 Strawweight
Women: 12 Atomweight, 13 Strawweight, 14 Flyweight, 15 Bantamweight, 16 Featherweight+
Exclude from per-division validation: -1 / -2 (P4P), 11 / 17 (Dominance lists)
